# core

> litert model initialisation and tools

In [ ]:
#| default_exp core

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import json, re
from html import escape
from mimetypes import guess_type
from litert_lm import (Engine, Backend, Conversation, Session, Message, Contents,
                       Content, Role, ToolCall, ToolEventHandler, set_min_log_severity)
from litert_lm._messages import (Text, ImageBytes, ImageFile, AudioBytes, AudioFile,
                                 ToolResponse, normalize_message)
from huggingface_hub import hf_hub_download
from fastcore.all import Path, store_attr, patch, L, GetAttr, ifnone
from safepyrun import RunPython

In [ ]:
#| export
def mk_content(o):
    "Convert `o` to a litert `Content`."
    if isinstance(o, Content): return o
    if isinstance(o, str): return Text(o)
    if isinstance(o, bytes): return ImageBytes(o)
    if isinstance(o, Path):
        mime = guess_type(str(o))[0] or ''
        return AudioFile(str(o)) if mime.startswith('audio/') else ImageFile(str(o))
    raise TypeError(f"Unsupported content type: {type(o)}")

def mk_msg(content, role='user'):
    "Create a litert `Message` from str/bytes/list/dict/Message."
    if content is None: return None
    if isinstance(content, Message): return content
    if isinstance(content, dict): return Message(Role(content['role']), Contents.of(content['content']))
    parts = [mk_content(o) for o in content] if isinstance(content, list) else [mk_content(content)]
    return Message(Role(role), Contents(parts))

def mk_msgs(msgs):
    "Normalize a list of messages to litert message dicts."
    if not msgs: return []
    if not isinstance(msgs, list): msgs = [msgs]
    return [normalize_message(m if isinstance(m, (Message, dict)) else mk_msg(m)) for m in msgs]

In [ ]:
#| hide
from fastcore.test import test_eq
test_eq(mk_msg("hello").to_json(), {"role": "user", "content": [{"type": "text", "text": "hello"}]})
test_eq(mk_msg("hi", role="model").to_json()["role"], "model")
test_eq([x["role"] for x in mk_msgs(["a", mk_msg("b", role="model")])], ["user", "model"])
assert isinstance(mk_content("x"), Text)

In [ ]:
#| export
class UsageStats:
    "Token usage for a chat turn, fed by `conv.token_count` diffs."
    def __init__(self, prompt_tokens=0, completion_tokens=0, total_tokens=0, n=0): store_attr()

    def __add__(self, other):
        if other is None: return self
        return UsageStats(*[getattr(self, k) + getattr(other, k)
            for k in ('prompt_tokens', 'completion_tokens', 'total_tokens', 'n')])
    def __radd__(self, other): return self if other in (None, 0) else self.__add__(other)

    def __repr__(self):
        return ' | '.join([f"total={self.total_tokens:,}", f"in={self.prompt_tokens:,}",
                           f"out={self.completion_tokens:,}", f"turns={self.n}"])

    def fmt(self):
        "Markdown `<details>` token block."
        if not self.total_tokens: return ''
        return f"\n\n<details><summary>{self.total_tokens:,} tokens</summary>\n\n`{self!r}`\n\n</details>\n"

In [ ]:
#| hide
a = UsageStats(prompt_tokens=10, completion_tokens=5, total_tokens=15, n=1)
b = UsageStats(prompt_tokens=3, completion_tokens=2, total_tokens=20, n=1)
c = a + b
test_eq((c.prompt_tokens, c.completion_tokens, c.n), (13, 7, 2))
assert "in=13" in repr(c) and "out=7" in repr(c)
test_eq((a + None).prompt_tokens, 10)

In [ ]:
#| export
class ChatCallback(GetAttr):
    "Base chat callback; reads chat state via `GetAttr` (`self.turn_msg` -> `chat.turn_msg`)."
    order, _default, chat, run = 0, 'chat', None, True
    def __repr__(self): return type(self).__name__

def run_cbs(chat, event):
    "Dispatch `event` to enabled callbacks in `order`; forward any yielded stream items."
    for cb in chat.cbs.sorted('order'):
        if cb.run and hasattr(cb, event):
            r = getattr(cb, event)()
            if r is not None: yield from r

In [ ]:
#| hide
class _Dummy: pass
class _A(ChatCallback):
    order = 20
    def before_send(self): self.chat.log.append('A')
class _B(ChatCallback):
    order = 10
    def before_send(self):
        self.chat.log.append('B')
        yield {'text': 'from-B'}
d = _Dummy(); d.log = []
a, b = _A(), _B(); a.chat = d; b.chat = d
d.cbs = L(a, b)
out = list(run_cbs(d, 'before_send'))
test_eq(d.log, ['B', 'A'])
test_eq(out, [{'text': 'from-B'}])
test_eq(repr(a), '_A')

In [ ]:
#| export
def _resp_text(resp):
    "Join text parts of a litert response/chunk dict."
    c = resp.get('content', []) if isinstance(resp, dict) else ''
    if isinstance(c, str): return c
    return ''.join(p.get('text', '') for p in c if isinstance(p, dict) and p.get('type') == 'text')

def _tc_summary(name, args, result=None):
    "One-line `<code>` summary of a tool call."
    params = ', '.join(f"{k}={v!r}" for k, v in (args or {}).items())
    res = f" -> {result}" if result is not None else ''
    return '<code>' + escape(f"{name}({params}){res}") + '</code>'

def mk_tr_details(name, args, result, mx=2000):
    "`<details>` JSON block for a completed tool call."
    body = json.dumps({'call': {'function': name, 'arguments': args}, 'result': str(result)[:mx]}, indent=2)
    return f"\n\n<details><summary>{_tc_summary(name, args, result)}</summary>\n\n```json\n{body}\n```\n\n</details>\n\n"

class StreamFormatter:
    "Format a litert response stream to markdown."
    def __init__(self, mx=2000): self.outp = ''; store_attr()
    def format_item(self, o):
        "Format one litert chunk dict."
        res = _resp_text(o)
        for p in (o.get('content', []) if isinstance(o, dict) else []):
            if isinstance(p, dict) and p.get('type') == 'tool_call':
                res += f"\n- ⏳ {_tc_summary(p.get('name', ''), p.get('arguments', {}))}\n"
        self.outp += res
        return res
    def format_stream(self, rs):
        "Yield markdown strings for each chunk."
        for o in rs: yield self.format_item(o)

def display_stream(rs, **kwargs):
    "Markdown-display a litert stream via IPython."
    from IPython.display import display, Markdown
    fmt, md = StreamFormatter(**kwargs), ''
    h = display(Markdown(' '), display_id=True)
    for o in fmt.format_stream(rs):
        md += o
        if md: h.update(Markdown(md))
    return fmt

In [ ]:
#| hide
fmt = StreamFormatter()
test_eq(_resp_text({"role": "assistant", "content": [{"type": "text", "text": "hi"}]}), "hi")
test_eq(fmt.format_item({"content": [{"type": "text", "text": "hel"}]}), "hel")
test_eq(fmt.format_item({"content": [{"type": "text", "text": "lo"}]}), "lo")
test_eq(fmt.outp, "hello")
tc = {"content": [{"type": "tool_call", "name": "add", "arguments": {"a": 1}}]}
assert "add" in fmt.format_item(tc)

In [ ]:
#| export
_tool_reminder = ("\n<system-reminder>After every tool call result, briefly summarise in prose "
                  "what you found before continuing or calling another tool.</system-reminder>")

class ToolReminderCallback(ChatCallback):
    "Inject a tool-summary reminder into the outgoing message when tools are registered."
    order = 30
    def __init__(self, tool_reminder=_tool_reminder): store_attr()
    def before_send(self):
        if self.chat.tools and self.chat.turn_msg is not None:
            self.chat.turn_msg.contents.contents.append(Text(self.tool_reminder))

In [ ]:
#| hide
class _C:
    def __init__(self): self.tools = [1]; self.turn_msg = mk_msg("hi")
c = _C(); cb = ToolReminderCallback(); cb.chat = c
cb.before_send()
assert any('summarise' in (p.text or '') for p in c.turn_msg.contents.contents)
c2 = _C(); c2.tools = []; cb2 = ToolReminderCallback(); cb2.chat = c2
cb2.before_send()
test_eq(len(c2.turn_msg.contents.contents), 1)

In [ ]:
#| export
class ChatToolHandler(ToolEventHandler):
    "Bridge litert's in-engine tool loop to Chat callbacks and history."
    def __init__(self, chat): self.chat = chat
    def approve_tool_call(self, tool_call):
        self.chat.turn_tc = tool_call
        self.chat.hist.append({'role': 'model', 'tool_calls': [tool_call]})
        for _ in run_cbs(self.chat, 'before_tool_calls'): pass
        return True
    def process_tool_response(self, tool_response):
        self.chat.turn_tool_result = tool_response
        name = self.chat.turn_tc.get('function', {}).get('name', '')
        self.chat.hist.append({'role': 'tool', 'content': [
            {'type': 'tool_response', 'name': name, 'response': tool_response}]})
        for _ in run_cbs(self.chat, 'after_tool_calls'): pass
        return tool_response

In [ ]:
#| hide
class _Chat:
    def __init__(self): self.hist = []; self.cbs = L()
ch = _Chat()
h = ChatToolHandler(ch)
tc = {"function": {"name": "add", "arguments": {"a": 1, "b": 2}}}
test_eq(h.approve_tool_call(tc), True)
test_eq(ch.turn_tc, tc)
assert ch.hist[-1]["role"] in ("assistant", "model")
out = h.process_tool_response({"result": 3})
test_eq(out, {"result": 3})
test_eq(ch.hist[-1]["role"], "tool")

litertlm defaults.

In [ ]:
#| export
gemma4_e4b='litert-community/gemma-4-E4B-it-litert-lm'
gemma4_e2b='litert-community/gemma-4-E2B-it-litert-lm'
gemma4_12b='litert-community/gemma-4-12B-it-litert-lm'

In [ ]:
#| export

def get_model(model_id: str, model_path:str):
	if model_path and Path(model_path).exists(): return model_path
	return hf_hub_download(model_id, repo_type='model')

class Chat:
	def __init__(self,
        model_id:str=gemma4_e2b, # hf model id
        model_path:str=None, # local model path
        backend:Backend=Backend.CPU, # litert backend
        multimodal:bool=True,
        sp:str='',
        messages:list=None,
		tools:list=None,


        **kwargs
	):
		"Local Gemma engine over litert_lm; downloads the model on first use."
		assert model_path or model_id, "model_id or model_path must be provided"
		model_path = model_path or hf_hub_download(model_id, repo_type='model')
		self.engine = Engine(model_path, backend=backend, multimodal=multimodal)
		self.conv: Conversation|None = None





In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()